# Pass 5 — 96-Image-Only Bootstrap Correction

This notebook corrects the Pass 5 Grad-CAM bootstrap analysis.

**Primary inferential analysis:** threshold = 0.50 only, giving exactly 96 unique held-out test images (65 benign, 31 malignant; 71 correct, 25 incorrect).

Thresholds 0.60, 0.70 and 0.80 are retained only as separate sensitivity analyses. They are never pooled with threshold 0.50, because the same 96 images are evaluated repeatedly.

The notebook reads the existing Pass 5 attribution table and does not retrain the classifier or recompute Grad-CAM maps.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
N_BOOTSTRAP = 10_000
PRIMARY_THRESHOLD = 0.50

PROJECT_ROOT = Path("..")
PASS5_DIR = PROJECT_ROOT / "results" / "pass5_gradcam_attribution"
METRICS_CSV = PASS5_DIR / "tables" / "pass5_attribution_metrics_by_image.csv"
OUT_DIR = PASS5_DIR / "statistics_96_image_only"
FIG_DIR = PASS5_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

assert METRICS_CSV.exists(), f"Missing: {METRICS_CSV}"
df = pd.read_csv(METRICS_CSV)
print("Loaded:", METRICS_CSV)
print("Shape:", df.shape)
print(df.columns.tolist())


In [ ]:
def first_existing(candidates):
    return next((c for c in candidates if c in df.columns), None)

threshold_col = first_existing(["threshold", "cam_threshold", "gradcam_threshold"])
true_col = first_existing(["true_label", "true_class", "label", "y_true"])
pred_col = first_existing(["predicted_label", "pred_class", "y_pred"])
image_id_col = first_existing(["image_id", "image_path", "image", "filename", "file_name", "path"])

required = {
    "threshold": threshold_col,
    "true label": true_col,
    "predicted label": pred_col,
    "image identifier": image_id_col,
}
missing = [k for k, v in required.items() if v is None]
assert not missing, f"Missing required columns: {missing}"

METRICS = [
    "attribution_iou",
    "attribution_precision",
    "lesion_coverage",
    "attribution_to_lesion_area_ratio",
    "cam_mass_fraction_inside_lesion",
    "attribution_pixel_fraction",
]
available_metrics = [m for m in METRICS if m in df.columns]
assert available_metrics, "No expected attribution metrics were found."

print("Threshold:", threshold_col)
print("True label:", true_col)
print("Predicted label:", pred_col)
print("Image ID:", image_id_col)
print("Metrics:", available_metrics)


In [ ]:
# Primary analysis: threshold 0.50, exactly 96 unique test images
primary = df[
    np.isclose(pd.to_numeric(df[threshold_col], errors="coerce"), PRIMARY_THRESHOLD)
].copy()

assert len(primary) == 96, f"Expected 96 rows at threshold 0.50; found {len(primary)}."
assert primary[image_id_col].nunique() == 96, (
    f"Expected 96 unique images; found {primary[image_id_col].nunique()}."
)

primary["true_label_norm"] = pd.to_numeric(primary[true_col], errors="raise").astype(int)
primary["predicted_label_norm"] = pd.to_numeric(primary[pred_col], errors="raise").astype(int)
primary["prediction_status"] = np.where(
    primary["true_label_norm"] == primary["predicted_label_norm"],
    "correct", "incorrect"
)

class_counts = primary["true_label_norm"].value_counts().to_dict()
status_counts = primary["prediction_status"].value_counts().to_dict()

assert class_counts.get(0, 0) == 65, class_counts
assert class_counts.get(1, 0) == 31, class_counts
assert status_counts.get("correct", 0) == 71, status_counts
assert status_counts.get("incorrect", 0) == 25, status_counts

print("Primary 96-image integrity checks PASSED.")
print("Benign:", class_counts.get(0, 0), "| Malignant:", class_counts.get(1, 0))
print("Correct:", status_counts.get("correct", 0), "| Incorrect:", status_counts.get("incorrect", 0))


In [ ]:
def bootstrap_mean_ci(values, n_bootstrap=N_BOOTSTRAP, seed=SEED):
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(x), size=(n_bootstrap, len(x)))
    boot = x[idx].mean(axis=1)
    return float(x.mean()), float(np.quantile(boot, .025)), float(np.quantile(boot, .975))

def bootstrap_difference_ci(a, b, n_bootstrap=N_BOOTSTRAP, seed=SEED):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    a = a[np.isfinite(a)]
    b = b[np.isfinite(b)]
    rng = np.random.default_rng(seed)
    ia = rng.integers(0, len(a), size=(n_bootstrap, len(a)))
    ib = rng.integers(0, len(b), size=(n_bootstrap, len(b)))
    boot = a[ia].mean(axis=1) - b[ib].mean(axis=1)
    observed = float(a.mean() - b.mean())
    return observed, float(np.quantile(boot, .025)), float(np.quantile(boot, .975))


In [ ]:
# Overall corrected 96-image bootstrap
rows = []
for metric in available_metrics:
    mean, lo, hi = bootstrap_mean_ci(primary[metric], seed=100)
    rows.append({
        "group": "overall", "metric": metric, "n_images": 96,
        "mean": mean, "ci95_low": lo, "ci95_high": hi,
        "threshold": PRIMARY_THRESHOLD, "bootstrap_replicates": N_BOOTSTRAP, "seed": SEED
    })

overall_ci = pd.DataFrame(rows)
overall_path = OUT_DIR / "pass5_96image_overall_bootstrap_ci.csv"
overall_ci.to_csv(overall_path, index=False)
display(overall_ci)


In [ ]:
# Benign vs malignant: separate image-level bootstrap CIs and difference
benign = primary[primary["true_label_norm"] == 0]
malignant = primary[primary["true_label_norm"] == 1]

rows = []
diff_rows = []

for metric in available_metrics:
    for group, g, seed in [
        ("benign", benign, 200),
        ("malignant", malignant, 201),
    ]:
        mean, lo, hi = bootstrap_mean_ci(g[metric], seed=seed)
        rows.append({
            "group": group, "metric": metric, "n_images": len(g),
            "mean": mean, "ci95_low": lo, "ci95_high": hi,
            "threshold": PRIMARY_THRESHOLD, "bootstrap_replicates": N_BOOTSTRAP, "seed": SEED
        })

    d, lo, hi = bootstrap_difference_ci(
        malignant[metric], benign[metric], seed=300
    )
    diff_rows.append({
        "comparison": "malignant_minus_benign", "metric": metric,
        "n_malignant": len(malignant), "n_benign": len(benign),
        "difference_mean": d, "ci95_low": lo, "ci95_high": hi,
        "threshold": PRIMARY_THRESHOLD, "bootstrap_replicates": N_BOOTSTRAP, "seed": SEED
    })

class_ci = pd.DataFrame(rows)
class_diff = pd.DataFrame(diff_rows)

class_ci.to_csv(OUT_DIR / "pass5_96image_class_bootstrap_ci.csv", index=False)
class_diff.to_csv(OUT_DIR / "pass5_96image_malignant_minus_benign_bootstrap_difference.csv", index=False)

display(class_ci)
display(class_diff)


In [ ]:
# Correct vs incorrect: separate image-level bootstrap CIs and difference
correct = primary[primary["prediction_status"] == "correct"]
incorrect = primary[primary["prediction_status"] == "incorrect"]

rows = []
diff_rows = []

for metric in available_metrics:
    for group, g, seed in [
        ("correct", correct, 400),
        ("incorrect", incorrect, 401),
    ]:
        mean, lo, hi = bootstrap_mean_ci(g[metric], seed=seed)
        rows.append({
            "group": group, "metric": metric, "n_images": len(g),
            "mean": mean, "ci95_low": lo, "ci95_high": hi,
            "threshold": PRIMARY_THRESHOLD, "bootstrap_replicates": N_BOOTSTRAP, "seed": SEED
        })

    d, lo, hi = bootstrap_difference_ci(
        incorrect[metric], correct[metric], seed=500
    )
    diff_rows.append({
        "comparison": "incorrect_minus_correct", "metric": metric,
        "n_incorrect": len(incorrect), "n_correct": len(correct),
        "difference_mean": d, "ci95_low": lo, "ci95_high": hi,
        "threshold": PRIMARY_THRESHOLD, "bootstrap_replicates": N_BOOTSTRAP, "seed": SEED
    })

status_ci = pd.DataFrame(rows)
status_diff = pd.DataFrame(diff_rows)

status_ci.to_csv(OUT_DIR / "pass5_96image_correct_incorrect_bootstrap_ci.csv", index=False)
status_diff.to_csv(OUT_DIR / "pass5_96image_incorrect_minus_correct_bootstrap_difference.csv", index=False)

display(status_ci)
display(status_diff)


In [ ]:
# Threshold sensitivity: each threshold analysed separately; never pooled
threshold_rows = []

for threshold in [0.50, 0.60, 0.70, 0.80]:
    subset = df[
        np.isclose(pd.to_numeric(df[threshold_col], errors="coerce"), threshold)
    ].copy()

    assert len(subset) == 96, f"Threshold {threshold}: expected 96 rows, got {len(subset)}."
    assert subset[image_id_col].nunique() == 96, (
        f"Threshold {threshold}: expected 96 unique images, "
        f"got {subset[image_id_col].nunique()}."
    )

    for metric in available_metrics:
        mean, lo, hi = bootstrap_mean_ci(
            subset[metric], seed=600 + int(threshold * 100)
        )
        threshold_rows.append({
            "threshold": threshold, "metric": metric, "n_images": 96,
            "mean": mean, "ci95_low": lo, "ci95_high": hi,
            "bootstrap_replicates": N_BOOTSTRAP, "seed": SEED,
            "analysis_role": "primary" if threshold == 0.50 else "sensitivity"
        })

threshold_ci = pd.DataFrame(threshold_rows)
threshold_ci.to_csv(
    OUT_DIR / "pass5_96image_threshold_sensitivity_bootstrap_ci.csv",
    index=False
)
display(threshold_ci)


In [ ]:
# Save an explicit analysis manifest documenting the correction
manifest = {
    "analysis": "Pass 5 corrected 96-image-only bootstrap",
    "source_table": str(METRICS_CSV),
    "primary_threshold": 0.50,
    "primary_n_images": 96,
    "benign_n": 65,
    "malignant_n": 31,
    "correct_n": 71,
    "incorrect_n": 25,
    "bootstrap_replicates": N_BOOTSTRAP,
    "seed": SEED,
    "inference_unit": "unique held-out test image",
    "pooled_384_row_analysis_used_as_primary": False,
    "thresholds_060_080": "separate sensitivity analyses only",
    "guardrail": (
        "Bootstrap CIs quantify sampling uncertainty around attribution summaries; "
        "they do not establish causal explanation, clinical validity, or independent lesion detection."
    )
}

(OUT_DIR / "pass5_96image_analysis_manifest.json").write_text(
    json.dumps(manifest, indent=2)
)
print(json.dumps(manifest, indent=2))


In [ ]:
# Primary figure: overall metrics with 95% bootstrap CIs
x = np.arange(len(overall_ci))
m = overall_ci["mean"].to_numpy()
lo = m - overall_ci["ci95_low"].to_numpy()
hi = overall_ci["ci95_high"].to_numpy() - m

plt.figure(figsize=(10, 5.5))
plt.errorbar(x, m, yerr=[lo, hi], fmt="o", capsize=4)
plt.xticks(x, overall_ci["metric"], rotation=35, ha="right")
plt.ylabel("Mean metric")
plt.title("Grad-CAM Attribution — 96-Image Bootstrap (Threshold 0.50)")
plt.tight_layout()
plt.savefig(
    FIG_DIR / "fig_pass5_96image_overall_attribution_bootstrap_ci.png",
    dpi=300, bbox_inches="tight"
)
plt.show()


In [ ]:
# Figure: CAM mass fraction inside lesion by true class
metric = "cam_mass_fraction_inside_lesion"
if metric in available_metrics:
    p = class_ci[class_ci["metric"] == metric]
    x = np.arange(len(p))
    m = p["mean"].to_numpy()
    lo = m - p["ci95_low"].to_numpy()
    hi = p["ci95_high"].to_numpy() - m

    plt.figure(figsize=(7, 5))
    plt.errorbar(x, m, yerr=[lo, hi], fmt="o", capsize=5)
    plt.xticks(x, p["group"])
    plt.ylabel("CAM mass fraction inside lesion")
    plt.title("Grad-CAM Lesion Mass — Benign vs Malignant")
    plt.tight_layout()
    plt.savefig(
        FIG_DIR / "fig_pass5_96image_cam_mass_benign_vs_malignant_ci.png",
        dpi=300, bbox_inches="tight"
    )
    plt.show()


In [ ]:
# Figure: CAM mass fraction inside lesion by prediction status
if metric in available_metrics:
    p = status_ci[status_ci["metric"] == metric]
    x = np.arange(len(p))
    m = p["mean"].to_numpy()
    lo = m - p["ci95_low"].to_numpy()
    hi = p["ci95_high"].to_numpy() - m

    plt.figure(figsize=(7, 5))
    plt.errorbar(x, m, yerr=[lo, hi], fmt="o", capsize=5)
    plt.xticks(x, p["group"])
    plt.ylabel("CAM mass fraction inside lesion")
    plt.title("Grad-CAM Lesion Mass — Correct vs Incorrect")
    plt.tight_layout()
    plt.savefig(
        FIG_DIR / "fig_pass5_96image_cam_mass_correct_vs_incorrect_ci.png",
        dpi=300, bbox_inches="tight"
    )
    plt.show()


## Manuscript methods wording

> “For inferential summaries of Grad-CAM attribution, bootstrap resampling was performed at the image level using the 96 held-out test images and a fixed attribution threshold of 0.50. This avoided treating multiple visualization thresholds of the same image as independent observations. Thresholds of 0.60, 0.70, and 0.80 were evaluated separately as sensitivity analyses.”

The resulting confidence intervals describe sampling uncertainty in the observed attribution summaries. They should not be interpreted as evidence of causal feature attribution, clinical validity, or independent lesion localization.
